In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import json
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt

from anngeno import AnnGeno
from scripts import get_burdens
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(num_cores)

In [ ]:
gene_id = 'ENSG00000106633'
phenotype = 'glycated_haemoglobin_hba1c'

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

## Get phenotypes and PRS

In [ ]:
phenos = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/phenotypes190_missing80_unique2.parquet').select(['individual', phenotype]).drop_nulls()
prs = pl.read_parquet('/home/dnanexus/data_dir/phenotypes/PRS190_missing80_unique2.parquet').select(['individual', f'{phenotype}_prs']).drop_nulls()

cov_list = config.get("covariates")

cov_df = pl.read_parquet(
    '/home/dnanexus/data_dir/phenotypes/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'
    ).rename(
        {'eid':'individual'}
    ).select(
        ['individual'] + cov_list
    ).with_columns(
        pl.col('individual').cast(pl.Int64).alias('individual')
    )
cov_df

all_df = phenos.join(prs, on='individual', how='inner').join(cov_df, on='individual', how='inner')
all_pd = all_df.to_pandas()
all_pd

In [ ]:
# Correct for covariates and PRS
combined_df = pd.DataFrame(index=all_pd.index)

y = all_pd[phenotype]
X = all_pd.drop(columns=[phenotype])
X = sm.add_constant(X)  # Add a constant term for the intercept

# Fit the model
model = sm.OLS(y, X).fit()

# Save residuals
residuals = pd.Series(model.resid, index=combined_df.index, name=f'{phenotype}_residual')

p_wide = pl.DataFrame(pd.concat([all_pd[['individual']], residuals], axis=1)).with_columns(pl.col('individual').cast(pl.String).alias('individual'))
p_wide

In [ ]:
# Get pdf with many phenotypes and then melt
pheno_cols = p_wide.columns[1:]  # Exclude 'individual' column

pdf = p_wide.unpivot(
    index=['individual'], 
    on=pheno_cols, 
    variable_name='phenotype',
    value_name='pheno_value'
)

pdf

## Read AnnGeno and get genotype

In [ ]:
ag = AnnGeno('/home/dnanexus/data_dir/dms_coding.ag', mode='r', low_mem=True)
# ag._set_annotations(split_ann.lazy())

maf = config.get('maf', None)
# maf = 0.00001
variants_to_keep = ag.annotations.filter((pl.col('af_ukb') < maf)).select('id').collect()['id']
ag.subset_variants(set(variants_to_keep))

samples = ag.samples
samples

### Get genotype for a gene

In [ ]:
reg_dict = ag.get_region(gene_id)

geno = reg_dict['genotypes']
anno_df = reg_dict['annotations']
geno

In [ ]:
# Find where the genotype is 1
rows, cols = np.where(geno == 1)
# Build the melted DataFrame
het = pl.DataFrame({
    'id': np.array(anno_df['id'])[rows],
    'individual': np.array(samples)[cols],
    'genotype': 1  # since we filtered for 1s only
})

# Homozygous genotypes
rows, cols = np.where(geno == 2)
hom = pl.DataFrame({
    'id': np.array(anno_df['id'])[rows],
    'individual': np.array(samples)[cols],
    'genotype': 2  # since we filtered for 1s only
})

geno_melt = pl.concat([het, hom]).with_columns(
    pl.lit(gene_id).alias('region')
)
geno_melt

In [ ]:
geno_melt['genotype'].value_counts()

In [ ]:
geno_melt['id'].value_counts(sort=True)

In [ ]:
geno_melt.select(['id', 'genotype']).unique()['genotype'].value_counts()

### Join genotypes and phenotypes, filter for EUR ancestry

In [ ]:
gp_df = geno_melt.join(pdf, on='individual')

# Restrict to EUR ancestry
eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').with_columns(
    pl.col("eid").cast(pl.Utf8)
)
gp_df = gp_df.filter(pl.col('individual').is_in(eur_samples['eid'].to_list()))

# Remove homozygous variant carriers for now #FIXME!!!
gp_df = gp_df.filter(pl.col('genotype') == 1)

gp_df

# Join annotations

### Get computational scores

In [ ]:
all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

anno_df

### Get DMS scores

In [ ]:
# pg = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_name').is_in(['GCK']))
pg = pl.read_parquet('/home/dnanexus/data_dir/exp_data/250717_proteingym_SNP_DMS_scores_human_coding_genes.parquet').filter(pl.col('gene_id').is_in([gene_id]))
pg

In [ ]:
pg['file_name'].value_counts().sort('count', descending=True)

In [ ]:
(
    ggplot(pg, aes(x='dms_score', fill='file_name')) +
    geom_histogram(bins=100, alpha=0.5, position='identity') +
    theme_538()
)

In [ ]:
pg_wide = pg.with_columns(
    pl.when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2022_activity')
      .then(pl.lit('dms_activity'))
      .when(pl.col('file_name') == 'HXK4_HUMAN_Gersing_2023_abundance')
      .then(pl.lit('dms_abundance'))
      .otherwise(None)
      .alias('score_type')
).pivot(
    index='mutant',
    on='score_type',
    values='dms_score'
)

pg_wide

In [ ]:
anno_melt = anno_df.join(pg_wide, on='mutant', how='inner')

all_annotation_list = list(set(all_annotation_list).intersection(set(anno_melt.columns)))
anno_melt = anno_melt.unpivot(
    index=['id', 'region', 'af_ukb'],
    on=all_annotation_list,
    variable_name='annotation',
    value_name='score'
)

anno_melt

### Join all scores with geno_pheno dataframe and get average pheno per variant

In [ ]:
gpa_df = gp_df.join(anno_melt, on='id', how='inner')
gpa_df

In [ ]:
plot_df = gpa_df.group_by(['id', 'phenotype', 'region', 'annotation']).agg(
    pl.len().alias('n_individuals'),
    pl.col('pheno_value').mean().alias('mean_pheno_value'),
    pl.col('score').mean().alias('mean_score'),
).drop_nulls(subset=['mean_pheno_value', 'mean_score'])

plot_df

## Spearman (rank) Correlation plots

### scatter plot single annotation

In [ ]:
import sys
from IPython.display import display

def plot_correlation(plot_df, phenotype, gene_id, annotation, method='spearman'):
    # Filter and add ranks
    df_filtered = plot_df.filter(
        (pl.col('phenotype') == f'{phenotype}_residual') &
        (pl.col('region') == gene_id) &
        (pl.col('annotation') == annotation)
    ).with_columns([
        pl.col('mean_score').rank().alias('score_rank'),
        pl.col('mean_pheno_value').rank().alias('pheno_rank')
    ])

    # Determine columns to correlate and plot
    if method.lower() == 'spearman':
        x_col, y_col = 'score_rank', 'pheno_rank'
    elif method.lower() == 'pearson':
        x_col, y_col = 'mean_score', 'mean_pheno_value'
    else:  # Pearson
        sys.exit(f"Unrecognized method. Use 'spearman' or 'pearson'.")

    # Compute correlation using Polars
    corr = df_filtered.select([pl.corr(x_col, y_col, method='pearson')]).to_numpy()[0, 0]

    corr_text = f"{method.title()} r = {corr:.2f}"

    # Build plot
    rc_plot = (
        ggplot(df_filtered.to_pandas(), aes(x=x_col, y=y_col)) +
        geom_point(alpha=0.25) +
        geom_smooth(method='lm', se=True, color='darkred') +
        theme_538() +
        labs(
            x=f"{annotation} {'rank' if method.lower()=='spearman' else ''}",
            y=f"{phenotype} residual {'rank' if method.lower()=='spearman' else ''}"
        ) +
        annotate(
            'text',
            x=df_filtered[x_col].min(),
            y=df_filtered[y_col].max(),
            label=corr_text,
            ha='left',
            va='top',
            size=12
        ) +
        theme(figure_size=(5, 4))
    )

    return rc_plot


In [ ]:
corr_method = 'spearman'
am_plot = plot_correlation(plot_df, phenotype, gene_id, 'am_pathogenicity', method=corr_method)
act_plot = plot_correlation(plot_df, phenotype, gene_id, 'dms_activity', method=corr_method)
ab_plot = plot_correlation(plot_df, phenotype, gene_id, 'dms_abundance', method=corr_method)

display(am_plot)
display(act_plot)
display(ab_plot)

### Plot all annotation correlations

In [ ]:
corr_method = 'spearman'

# Group by annotation and compute correlation
corr_df = (
    plot_df
    .group_by(['annotation', 'phenotype', 'region'])
    .agg([
        pl.corr('mean_score', 'mean_pheno_value', method=corr_method).alias('corr')
    ])
    .with_columns([
        pl.col('corr').abs().alias('abs_corr')
    ])
    .filter(pl.col('abs_corr').is_not_null())
    .sort('abs_corr', descending=True)
)

corr_df

In [ ]:
# Convert to pandas
rank_corr_df = corr_df.drop_nans().to_pandas()
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)
rank_corr_df['color_dms'] = rank_corr_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot bar plot of correlations
(
    ggplot(rank_corr_df, aes(x='annotation', y='abs_corr', fill='color_dms')) +
    geom_col(alpha=0.8) +
    coord_flip() +  # Flip for better readability if many annotations
    theme_bw() +
    labs(
        x='Annotation',
        y='absolute Spearman correlation'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)

## Bootstraping across individuals

In [ ]:
def bootstrap_samples(
    gpa_df: pl.DataFrame,
    n_bootstraps: int,
    seed: int = None
) -> pl.DataFrame:
    if seed is not None:
        np.random.seed(seed)

    # Get unique individuals
    unique_ids = gpa_df.select('individual').unique().to_series().to_list()
    n_ids = len(unique_ids)

    # Prepare all bootstrap samples in one array
    sampled_ids = np.random.choice(unique_ids, size=(n_bootstraps, n_ids), replace=True)

    # Flatten and make a DataFrame with bootstrap_id
    boot_id_col = np.repeat(np.arange(n_bootstraps), n_ids)
    sampled_flat = pl.LazyFrame({
        'bootstrap_id': boot_id_col,
        'individual': sampled_ids.ravel()
    })

    # Lazy join to replicate rows
    boot_df = sampled_flat.join(
        gpa_df.lazy(), on='individual', how='left'
    )

    # Group by bootstrap_id + original grouping columns
    result = (
        boot_df
        .group_by(['bootstrap_id', 'id', 'phenotype', 'region', 'annotation'])
        .agg([
            pl.len().alias('n_individuals'),
            pl.col('pheno_value').mean().alias('mean_pheno_value'),
            pl.col('score').mean().alias('mean_score'),
        ])
        .collect()
    )

    return result

In [ ]:
%%time

boot_df = bootstrap_samples(gpa_df, n_bootstraps=1000, seed=42)
boot_df

In [ ]:
corr_df = (
    boot_df
    .group_by(['phenotype', 'region', 'annotation', 'bootstrap_id'])
    .agg([
        pl.corr('mean_score', 'mean_pheno_value', method=corr_method).alias('corr')
    ])
    .with_columns([
        pl.col('corr').abs().alias('abs_corr')
    ])
    .sort('abs_corr', descending=True)
    .drop_nans()
)
corr_df

In [ ]:
corr_method = 'spearman'

# Group by annotation and compute correlation
corr_df = (
    boot_df
    .group_by(['phenotype', 'region', 'annotation', 'bootstrap_id'])
    .agg([
        pl.corr('mean_score', 'mean_pheno_value', method=corr_method).alias('corr')
    ])
    .with_columns([
        pl.col('corr').abs().alias('abs_corr')
    ])
    .sort('abs_corr', descending=True)
    .drop_nans()
)

summary_df = (
    corr_df
    .group_by("annotation")
    .agg([
        pl.col("abs_corr").mean().alias("mean_abs_corr"),
        pl.col("abs_corr").std().alias("se_abs_corr"),
        pl.col("abs_corr").quantile(0.025).alias("quant_low"),
        pl.col("abs_corr").quantile(0.975).alias("quant_high"),
    ])
    .with_columns([
        (pl.col("mean_abs_corr") - 1.96*pl.col("se_abs_corr")).alias("ci_low"),
        (pl.col("mean_abs_corr") + 1.96*pl.col("se_abs_corr")).alias("ci_high")
    ])
    .to_pandas()
)

summary_df

In [ ]:
# Ordering by mean correlation
summary_df['annotation'] = pd.Categorical(
    summary_df['annotation'],
    categories=summary_df.sort_values('mean_abs_corr', ascending=False)['annotation'],
    ordered=True
)

summary_df['color_dms'] = summary_df['annotation'].str.startswith('dms_').map({True: 'DMS', False: 'Other'})

# Plot
(
    ggplot(summary_df, aes(x='annotation', y='mean_abs_corr', fill='color_dms')) +
    geom_col(alpha=0.8) +
    # geom_errorbar(aes(ymin='ci_low', ymax='ci_high'), width=0.2) +
    geom_errorbar(aes(ymin='quant_low', ymax='quant_high'), width=0.2) +
    coord_flip() +
    theme_538() +
    labs(
        x='Annotation',
        y='Mean absolute correlation (95% CI)'
    ) +
    theme(
        figure_size=(8, 3),
        legend_position='none'
    )
)